# Trabalho Prático — livros do Goodreads

**Integrantes:** João Henrique Valbusa Lima - 2023100583 - joao.h.lima@edu.ufes.br

Nesta versão fazemos as etapas 1 a 4: conhecer os dados, selecionar 1% dos livros, limpar as descrições e construir a matriz TF-IDF. O gênero será usado somente na análise dos agrupamentos, em uma etapa posterior.

Abra o notebook na pasta `trabalhoPratico`, selecione o kernel **Python — UFES CD** e execute as células em ordem. Se faltar alguma biblioteca, instale com `%pip install -r requirements.txt`.

A preparação do notebook contou com auxílio do Codex (OpenAI); revisar e mencionar a colaboração conforme o enunciado.

## Bibliotecas

Usaremos Pandas e NLTK, como nas aulas. Os dois downloads são recursos do NLTK: o tokenizador e a lista de stopwords. A primeira execução precisa de internet.

In [1]:
import string
import pandas as pd
import nltk
from nltk import word_tokenize
from nltk.corpus import stopwords

nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

/home/chefe/workspace/UFES/periodo-8/CD/.venv/lib/python3.12/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/chefe/workspace/UFES/periodo-8/CD/.venv/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):


True

## 1. Conhecer os dados

Referência: `../aulas/aula_pandas.ipynb`, nas partes de leitura de CSV, informações do DataFrame e descrição básica dos dados.

In [2]:
dados = pd.read_csv('goodreads_books.csv')
dados.head()

,book_id,title,author,description,genre,publication_year,average_rating,ratings_count,num_pages
0,32941127,Black Panther: A Nation Under Our Feet,Ta-Nehisi Coates,A new era for the Black Panther begins as the ...,graphic-novel|fantasy|science-fiction|literary...,2017.0,4.25,52,344.0
1,22929618,the Beastly Pirates,John Kelly,They're called The Beastly Pirates and this fe...,children,2015.0,3.56,8,32.0
2,13550502,"The Ill-Made Knight (William Gold, #1)",Christian Cameron,William Gold comes into the world as his famil...,historical-fiction|literary-fiction|fantasy|no...,2013.0,4.37,790,457.0
3,16059950,The Mystic Princesses and the Whirlpool,P.J. LaRue,Meet The Mystic Princesses - Role models who s...,children|fantasy,2012.0,4.33,9,34.0
4,6684834,88 Short Stories,Guy de Maupassant,Contents: 1. Yvette (Yvette) 2. Our Friends th...,short-stories,1950.0,4.50,2,815.0


In [3]:
print('Quantidade de livros:', len(dados))
dados.info()

Quantidade de livros: 25000
<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   book_id           25000 non-null  int64  
 1   title             25000 non-null  str    
 2   author            25000 non-null  str    
 3   description       25000 non-null  str    
 4   genre             25000 non-null  str    
 5   publication_year  19931 non-null  float64
 6   average_rating    25000 non-null  float64
 7   ratings_count     25000 non-null  int64  
 8   num_pages         17727 non-null  float64
dtypes: float64(3), int64(2), str(4)
memory usage: 1.7 MB


In [4]:
dados.isna().sum()

book_id                0
title                  0
author                 0
description            0
genre                  0
publication_year    5069
average_rating         0
ratings_count          0
num_pages           7273
dtype: int64

In [5]:
print('IDs repetidos:', dados['book_id'].duplicated().sum())
print('Descrições repetidas:', dados['description'].duplicated().sum())
dados['description'].str.len().describe()

IDs repetidos: 0
Descrições repetidas: 237


count    25000.000000
mean       830.876680
std        531.725617
min        100.000000
25%        475.000000
50%        753.000000
75%       1056.000000
max       9398.000000
Name: description, dtype: float64

Ano de publicação e número de páginas podem ter valores ausentes. Vamos mantê-los, pois só a descrição será usada para agrupar os livros. Sinopses repetidas também são mantidas: podem pertencer a edições ou livros diferentes.

Antes da amostragem, retiramos apenas linhas sem descrição ou com descrição em branco. O CSV original permanece intacto.

In [6]:
dados = dados.dropna(subset=['description'])
dados = dados[dados['description'].str.strip() != '']
print('Livros com descrição:', len(dados))

Livros com descrição: 25000


## 2. Selecionar 1% dos livros

`frac=0.01` seleciona 1% das linhas. `random_state` fixa o sorteio, e `replace=False` impede selecionar a mesma linha duas vezes. Para executar com todos os dados depois, troque `fracao` por `1.0`.

A amostra de 1% serve para desenvolver o código. As conclusões finais serão feitas após a execução em uma base maior.

In [7]:
fracao = 0.01
semente = 67

amostra = dados.sample(frac=fracao, random_state=semente, replace=False).copy()
print('Livros selecionados:', len(amostra))
amostra[['book_id', 'description']].head()

Livros selecionados: 250


,book_id,description
19222,16131414,Teen rocker Ritchie Sudden is pretty sure his ...
20603,22890662,The hilarious fifth installment of the #1New Y...
16401,24004414,Oliver has a horrifying vision of the future a...
4127,4723837,Convinced that her family is poor because her ...
4116,3782687,"Ninianee and Erianthee, the daughters of Nimua..."


## 3. Limpar as descrições

Referência: `../aulas/aula_dados_textuais.ipynb`, de “Dividindo o texto em sentenças e palavras” até “Removendo stop words”.

Seguiremos os passos da aula: tokenização, minúsculas, remoção de pontuação, seleção de palavras com `isalpha()` e remoção de stopwords. Vamos primeiro observar uma descrição e depois colocar esses passos em uma função.

Usaremos a lista inglesa do NLTK porque a maioria das sinopses está em inglês. Preservaremos `no`, `nor` e `not` para manter essas negações. Esta versão não faz identificação automática de idioma: sinopses em outros idiomas precisam ser revisadas antes de interpretar os agrupamentos, pois podem formar grupos pelo idioma. Também não aplicaremos stemming nem conversão de acentos nesta primeira versão.

In [8]:
stop_words = set(stopwords.words('english'))
stop_words = stop_words - {'no', 'nor', 'not'}
tabela = str.maketrans('', '', string.punctuation)

texto = amostra['description'].iloc[0]
print(texto)

Teen rocker Ritchie Sudden is pretty sure his life has just jumped the shark. Except he hates being called a teen, his band doesn't play rock, and "jumping the shark" is yet another dumb cliche. Part of Ritchie wants to drop everything and walk away. Especially the part that's serving ninety days in a juvenile detention center. Telling the story of the year leading up to his arrest, Ritchie grabs readers by the throat before (politely) inviting them along for the (max-speed) ride. A battle of the bands looms. Dad split about five minutes before Mom's girlfriend moved in. There's the matter of trying to score with the dangerously hot Ravenna Woods while avoiding the dangerously huge Spence Proffer--not to mention just trying to forget what his sister, Beth, said the week before she died. This latest offering from acclaimed author Sean Beaudoin is alternately raw, razor-sharp, and genuinely hilarious.


In [9]:
tokens = word_tokenize(texto)
tokens = [palavra.lower() for palavra in tokens]
tokens = [palavra.translate(tabela) for palavra in tokens]
palavras = [palavra for palavra in tokens if palavra.isalpha()]
palavras = [palavra for palavra in palavras if palavra not in stop_words]
palavras[:40]

['teen',
 'rocker',
 'ritchie',
 'sudden',
 'pretty',
 'sure',
 'life',
 'jumped',
 'shark',
 'except',
 'hates',
 'called',
 'teen',
 'band',
 'nt',
 'play',
 'rock',
 'jumping',
 'shark',
 'yet',
 'another',
 'dumb',
 'cliche',
 'part',
 'ritchie',
 'wants',
 'drop',
 'everything',
 'walk',
 'away',
 'especially',
 'part',
 'serving',
 'ninety',
 'days',
 'juvenile',
 'detention',
 'center',
 'telling',
 'story']

A função abaixo reúne exatamente os passos anteriores. `apply` executa a função para cada descrição, mantendo o texto original em `description` e criando `description_clean`. Não usamos gênero, título ou autor na limpeza.

In [10]:
def limpar_texto(texto):
    tokens = word_tokenize(texto)
    tokens = [palavra.lower() for palavra in tokens]
    tokens = [palavra.translate(tabela) for palavra in tokens]
    palavras = [palavra for palavra in tokens if palavra.isalpha()]
    palavras = [palavra for palavra in palavras if palavra not in stop_words]
    return ' '.join(palavras)

amostra['description_clean'] = amostra['description'].apply(limpar_texto)
amostra[['description', 'description_clean']].head()

,description,description_clean
19222,Teen rocker Ritchie Sudden is pretty sure his ...,teen rocker ritchie sudden pretty sure life ju...
20603,The hilarious fifth installment of the #1New Y...,hilarious fifth installment york timesbestsell...
16401,Oliver has a horrifying vision of the future a...,oliver horrifying vision future recovers first...
4127,Convinced that her family is poor because her ...,convinced family poor parents refuse get horse...
4116,"Ninianee and Erianthee, the daughters of Nimua...",ninianee erianthee daughters nimuar duz vildec...


In [11]:
print('Textos vazios após a limpeza:', (amostra['description_clean'] == '').sum())
amostra['description_clean'].str.split().str.len().describe()

Textos vazios após a limpeza: 0


count    250.000000
mean      78.004000
std       51.437918
min       10.000000
25%       43.000000
50%       69.000000
75%      101.000000
max      298.000000
Name: description_clean, dtype: float64

## Salvar a amostra preparada

O arquivo abaixo guarda as linhas selecionadas e a nova coluna de texto limpo. As demais colunas são mantidas para consulta posterior; somente `description_clean` deverá ser a entrada do TF-IDF. `genre` não pode ser usado para construir os grupos.

Os resultados da versão anterior continuam na pasta `resultados`. O notebook anterior está guardado em `resultados/versao_anterior`. Esta versão simplificada gera apenas o CSV abaixo.

In [12]:
amostra.to_csv('amostra_preprocessada.csv', index=False)
print('Amostra salva em amostra_preprocessada.csv')

Amostra salva em amostra_preprocessada.csv


## 4. Construir a matriz TF-IDF

Referência: `../aulas/aula_dados_textuais.ipynb`, seção “Matriz de TF-IDF”.

O TF-IDF representa cada descrição por números: palavras frequentes em uma descrição e pouco comuns nas demais tendem a receber mais peso. Usaremos `TfidfVectorizer`, como no exemplo da aula, com a configuração padrão (palavras individuais).

`fit_transform` aprende o vocabulário e os pesos IDF usando as descrições limpas da amostra. Não usamos os gêneros. Cada linha de `X` corresponde a um livro, na mesma ordem de `amostra`, e cada coluna corresponde a um termo. Para executar com todos os dados, mude `fracao` e execute novamente desde o início, incluindo o ajuste do TF-IDF.

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(amostra['description_clean'])

print('Livros:', X.shape[0])
print('Termos no vocabulário:', X.shape[1])

Livros: 250
Termos no vocabulário: 7390


Vamos visualizar somente as primeiras 5 linhas e 10 colunas. Mantemos `X` como matriz esparsa, que armazena os valores não nulos e economiza memória. Convertemos apenas esse pequeno recorte em uma tabela.

In [14]:
termos = vectorizer.get_feature_names_out()

pd.DataFrame(
    X[:5, :10].toarray(),
    index=amostra['book_id'].iloc[:5],
    columns=termos[:10]
)

,abandon,abandoned,abduction,abilities,ability,able,abnegation,abner,aboard,aboulela
book_id,,,,,,,,,,
16131414,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
22890662,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
24004414,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4723837,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3782687,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Para entender os pesos, podemos observar os 10 termos com maior TF-IDF na primeira descrição. Os valores representam pesos de termos, não probabilidades de gênero.

In [15]:
pesos = pd.Series(X[0].toarray().ravel(), index=termos)
pesos.sort_values(ascending=False).head(10)

ritchie        0.328267
shark          0.218845
dangerously    0.203631
teen           0.184463
part           0.154878
trying         0.143240
ravenna        0.109422
sean           0.109422
juvenile       0.109422
week           0.109422
dtype: float64

A matriz `X` está pronta para a próxima etapa: redução de dimensionalidade com PCA/SVD. Antes da entrega, registrar as especificações do computador, revisar as sinopses em outros idiomas e explicar as decisões de limpeza.